# Lecture 14 — Open Systems & Nonlinear PDEs (numpy companion)

Two hurdles separate realistic problems from textbook unitary quantum simulation:

1. **Open-system dynamics** are *non-unitary*. A Markovian open system obeys the
   **Lindblad master equation**
   $$\dot\rho = -\mathrm{i}[H,\rho] + \sum_k \Big(L_k\rho L_k^\dagger - \tfrac12\{L_k^\dagger L_k,\rho\}\Big),$$
   which contracts coherences and populations. Quantum algorithms handle it by
   *dilation* (Stinespring) or Schrödingerization; here we integrate it classically
   to expose the physics an algorithm must reproduce.
2. **Nonlinear PDEs** cannot be loaded directly onto a linear quantum evolution. Some
   admit an exact *linearizing transform*: the **Cole–Hopf** map turns the nonlinear
   viscous **Burgers** equation into the linear **heat** equation.

We verify both facts numerically (pure numpy, no dedicated algorithm class):
(a) single-qubit **dephasing** and **amplitude damping** match their analytic decay
laws; (b) the Cole–Hopf transform of a heat solution satisfies Burgers' equation to
finite-difference accuracy.

Contents:
1. Setup and a master-equation integrator
2. Dephasing: off-diagonal decay $\propto e^{-2\gamma t}$
3. Amplitude damping: population decay $\propto e^{-\gamma t}$
4. Cole–Hopf: heat $\to$ Burgers
5. Your turn

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive")

try:
    from scipy.linalg import expm as _expm
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False
    def _expm(M):
        # general (non-normal) small matrix exponential via eigdecomposition fallback
        w, V = np.linalg.eig(M)
        return (V * np.exp(w)) @ np.linalg.inv(V)

# Pauli operators and a couple of jump operators.
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
sminus = np.array([[0, 1], [0, 0]], dtype=complex)   # |0><1|, lowering to ground |0>

def lindbladian(H, jumps):
    '''Dense Liouvillian superoperator L such that vec(rho_dot) = L @ vec(rho),
    built by applying the generator to each basis matrix (convention-independent).'''
    d = H.shape[0]; D = d * d
    S = np.zeros((D, D), dtype=complex)
    for i in range(D):
        e = np.zeros(D, dtype=complex); e[i] = 1.0
        rho = e.reshape(d, d)
        drho = -1j * (H @ rho - rho @ H)
        for Lk in jumps:
            Ld = Lk.conj().T
            drho += Lk @ rho @ Ld - 0.5 * (Ld @ Lk @ rho + rho @ Ld @ Lk)
        S[:, i] = drho.reshape(-1)
    return S

np.set_printoptions(precision=4, suppress=True)
print("scipy available:", _HAVE_SCIPY)

scipy available: True


## 2. Dephasing: off-diagonal decay $\propto e^{-2\gamma t}$

Pure dephasing has a single Hermitian jump operator $L=\sqrt{\gamma}\,Z$ and $H=0$.
Applying the Lindblad generator to the coherence element gives
$\dot\rho_{01} = -2\gamma\,\rho_{01}$, so starting from $\lvert+\rangle\!\langle+\rvert$
(coherence $\tfrac12$) we predict $\lvert\rho_{01}(t)\rvert = \tfrac12 e^{-2\gamma t}$.

**Tolerance:** matches analytic $e^{-2\gamma t}$ decay to $10^{-9}$.

In [2]:
gamma = 0.6
S_deph = lindbladian(np.zeros((2, 2), dtype=complex), [np.sqrt(gamma) * Z])
rho0 = 0.5 * np.ones((2, 2), dtype=complex)   # |+><+|

times = np.linspace(0, 3, 25)
coh = []
for t in times:
    rho_t = (_expm(S_deph * t) @ rho0.reshape(-1)).reshape(2, 2)
    coh.append(abs(rho_t[0, 1]))
coh = np.array(coh)
analytic = 0.5 * np.exp(-2 * gamma * times)

max_dev = np.max(np.abs(coh - analytic))
print("max |coherence - 0.5 e^{-2 gamma t}|:", format(max_dev, ".3e"))
# trace preservation at the final time
rho_end = (_expm(S_deph * times[-1]) @ rho0.reshape(-1)).reshape(2, 2)
print("trace at t_end:", round(np.trace(rho_end).real, 12))
assert max_dev < 1e-9
assert np.isclose(np.trace(rho_end).real, 1.0, atol=1e-9)
print("Verified: dephasing coherence follows 0.5 e^{-2 gamma t} (dev < 1e-9), trace preserved.")

max |coherence - 0.5 e^{-2 gamma t}|: 5.551e-17
trace at t_end: 1.0
Verified: dephasing coherence follows 0.5 e^{-2 gamma t} (dev < 1e-9), trace preserved.


## 3. Amplitude damping: population decay $\propto e^{-\gamma t}$

Amplitude damping models spontaneous emission: a single jump $L=\sqrt{\gamma}\,\sigma_-$
with $\sigma_-=\lvert0\rangle\!\langle1\rvert$. Starting in the excited state
$\lvert1\rangle\!\langle1\rvert$, the excited population decays as $e^{-\gamma t}$ (the
$T_1$ law) and probability flows into the ground state.

**Tolerance:** matches analytic $e^{-\gamma t}$ to $10^{-9}$.

In [3]:
S_amp = lindbladian(np.zeros((2, 2), dtype=complex), [np.sqrt(gamma) * sminus])
rho_exc = np.array([[0, 0], [0, 1]], dtype=complex)   # |1><1|

pops = []
for t in times:
    rho_t = (_expm(S_amp * t) @ rho_exc.reshape(-1)).reshape(2, 2)
    pops.append(rho_t[1, 1].real)
pops = np.array(pops)
analytic_pop = np.exp(-gamma * times)

max_dev_amp = np.max(np.abs(pops - analytic_pop))
print("max |excited population - e^{-gamma t}|:", format(max_dev_amp, ".3e"))
rho_end2 = (_expm(S_amp * times[-1]) @ rho_exc.reshape(-1)).reshape(2, 2)
print("trace at t_end:", round(np.trace(rho_end2).real, 12))
assert max_dev_amp < 1e-9
assert np.isclose(np.trace(rho_end2).real, 1.0, atol=1e-9)
print("Verified: amplitude-damping population follows e^{-gamma t} (dev < 1e-9), trace preserved.")

max |excited population - e^{-gamma t}|: 1.665e-16
trace at t_end: 1.0
Verified: amplitude-damping population follows e^{-gamma t} (dev < 1e-9), trace preserved.


In [4]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(times, coh, "o", label="dephasing coherence (numeric)")
ax.plot(times, analytic, "-", label=r"$\frac{1}{2} e^{-2\gamma t}$")
ax.plot(times, pops, "s", label="amp-damping excited pop (numeric)")
ax.plot(times, analytic_pop, "-", label=r"$e^{-\gamma t}$")
ax.set_xlabel("time  t"); ax.set_ylabel("value")
ax.set_title(r"Open-system decay, $\gamma = 0.6$")
ax.legend(fontsize=8); ax.grid(True, ls=":")
plt.tight_layout(); plt.show()
print("numeric markers lie on the analytic curves.")

numeric markers lie on the analytic curves.


## 4. Cole–Hopf: heat $\to$ Burgers

The viscous Burgers equation $u_t + u\,u_x = \nu\,u_{xx}$ is nonlinear, but the
**Cole–Hopf** substitution $u = -2\nu\,\phi_x/\phi$ maps it *exactly* onto the linear
heat equation $\phi_t = \nu\,\phi_{xx}$. We take a known heat solution
$\phi(x,t) = 1 + \tfrac12 e^{-\nu t}\cos x$ (a decaying Fourier mode), form the
corresponding $u$, and check the Burgers residual $u_t + u\,u_x - \nu\,u_{xx}$ on a
periodic grid via finite differences.

**Tolerance:** the PDE residual is $\sim 0$ (finite-difference limited, $< 10^{-4}$).

In [5]:
nu = 0.3
Nx = 400
Lx = 2 * np.pi
xg = np.linspace(0, Lx, Nx, endpoint=False)
dxg = xg[1] - xg[0]

def phi(x, t):        # solves phi_t = nu phi_xx  (single cosine mode)
    return 1.0 + 0.5 * np.exp(-nu * t) * np.cos(x)
def phi_x(x, t):
    return -0.5 * np.exp(-nu * t) * np.sin(x)
def u_field(x, t):    # Cole-Hopf transform
    return -2 * nu * phi_x(x, t) / phi(x, t)

# Sanity: phi indeed satisfies the heat equation (analytic derivatives).
def phi_xx(x, t): return -0.5 * np.exp(-nu * t) * np.cos(x)
def phi_t(x, t):  return -nu * 0.5 * np.exp(-nu * t) * np.cos(x)
heat_resid = np.max(np.abs(phi_t(xg, 0.5) - nu * phi_xx(xg, 0.5)))
print("heat-equation residual for phi:", format(heat_resid, ".3e"))
assert heat_resid < 1e-12

# Burgers residual for u via central differences (time and periodic space).
t0, dt = 0.5, 1e-4
u_now = u_field(xg, t0)
u_t = (u_field(xg, t0 + dt) - u_field(xg, t0 - dt)) / (2 * dt)
u_x = (np.roll(u_now, -1) - np.roll(u_now, 1)) / (2 * dxg)
u_xx = (np.roll(u_now, -1) - 2 * u_now + np.roll(u_now, 1)) / dxg**2
burgers_resid = u_t + u_now * u_x - nu * u_xx

max_resid = np.max(np.abs(burgers_resid))
print("max |Burgers residual|:", format(max_resid, ".3e"))
assert max_resid < 1e-4
print("Verified: Cole-Hopf u = -2 nu phi_x/phi solves Burgers (residual < 1e-4).")

heat-equation residual for phi: 2.776e-17
max |Burgers residual|: 1.603e-05
Verified: Cole-Hopf u = -2 nu phi_x/phi solves Burgers (residual < 1e-4).


## 5. Your turn
- Add a coherent drive $H = \tfrac{\Omega}{2}X$ to the amplitude-damping model and
  watch damped Rabi oscillations emerge in the excited population.
- Combine dephasing *and* amplitude damping (two jump operators) and confirm the
  coherence now decays at the combined rate $\gamma_\phi + \gamma_{\text{amp}}/2$.
- Sharpen the Cole–Hopf grid (larger `Nx`, smaller `dt`) and watch the Burgers
  residual shrink like $\Delta x^2 + \Delta t^2$ — evidence it is purely a
  discretization artifact, not a modeling error.

Lecture 15 opens Part IV: **tensor-network states**, where the singular-value
decomposition compresses many-body wavefunctions.